In [ ]:
# %pip install transformers torch
# %pip install transformers
# %pip install ipywidgets
# %pip install numpy==1.26.4
# %pip install datasets
# %pip install scipy
# %pip install sentencepiece
# %pip install scipy==1.12
# %pip install tf-keras
# %pip install 'accelerate>=0.26.0'
# %pip install --upgrade ipython

In [ ]:
from transformers import RobertaTokenizer, RobertaForCausalLM
from transformers import LlamaTokenizer, LlamaForCausalLM
from datasets import load_dataset
import torch
import tqdm

In [ ]:
# device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
device = 'cpu'
device

In [ ]:
datasets = load_dataset('wikitext', 'wikitext-2-raw-v1')
print(datasets["train"][10])

In [ ]:
from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [ ]:
show_random_elements(datasets["train"])

In [ ]:
# model_checkpoint = "distilgpt2"
# model_checkpoint = "roberta-base"

model_checkpoint = "NousResearch/Llama-2-7b-hf"



from transformers import AutoTokenizer
    
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)
tokenized_datasets = datasets.map(tokenize_function, batched=True, num_proc=4, remove_columns=["text"])

In [ ]:
tokenized_datasets["train"][1]

In [ ]:
block_size = 32

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
        # customize this part to your needs.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
)

In [ ]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(model_checkpoint, is_decoder=True)

In [ ]:
from transformers import Trainer, TrainingArguments

model_name = model_checkpoint.split("/")[-1]

training_args = TrainingArguments(
    f"{model_name}-finetuned-wikitext2",
    eval_strategy = "epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=20,
    logging_dir='./logs2',
    push_to_hub=False,
    use_cpu=True,
    bf16=True,
)

In [ ]:
trainer = Trainer(
    model=model.to(device),
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
)

In [ ]:
trainer.train()

In [ ]:
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")